In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Organizar los resultados en un DataFrame
df_simulacion = pd.read_csv("datos_prueba.csv", sep=";")
df_simulacion['Tiempo'] = 1

In [2]:
# Calcular el lambda estimado
total_eventos = df_simulacion['y'].sum()
total_tiempo = df_simulacion['Tiempo'].sum()

lambda_recuperado = total_eventos / total_tiempo

print(f"Lambda Recuperado: {lambda_recuperado:.4f}")

Lambda Recuperado: 1.3700


In [3]:
import statsmodels.api as sm

# El logaritmo del tiempo actúa como 'offset' en una regresión de Poisson
df_simulacion['log_tiempo'] = np.log(df_simulacion['Tiempo'])

# Ajustar el modelo
modelo = sm.GLM(df_simulacion['y'], 
                np.ones(len(df_simulacion)), # Solo intercepto
                family=sm.families.Poisson(), 
                offset=df_simulacion['log_tiempo']).fit()

# El valor recuperado es exp(intercepto)
lambda_glm = np.exp(modelo.params[0])
print(f"Lambda recuperado via GLM: {lambda_glm:.4f}")

Lambda recuperado via GLM: 1.3700


C:\Users\Athan\AppData\Local\Temp\ipykernel_27040\1377729082.py:13: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  lambda_glm = np.exp(modelo.params[0])


# 3. Establecer el valor de lambda (tasa de eventos por unidad de tiempo)
lam = 0.0876

# 4. Simular el número de eventos para cada paciente
# La media de la distribución de Poisson para cada paciente es lambda * t_i
eventos = np.random.poisson(lam)

# Histograma de eventos observados
plt.hist(eventos, bins=np.arange(eventos.max() + 2) - 0.5, 
         color='seagreen', edgecolor='white', rwidth=0.8)

plt.title(f'Distribución de Eventos (n={n}, $\lambda$ recup={lambda_recuperado:.3f})')
plt.xlabel('Número de Eventos por Paciente')
plt.ylabel('Frecuencia (Pacientes)')
plt.xticks(range(eventos.max() + 1))
plt.grid(axis='y', linestyle='--', alpha=0.7)

In [6]:
X1 = df_simulacion['x1']
X2 = df_simulacion['x2']
print("Tipo de X1:", X1.dtype)
print("Tipo de X2:", X2.dtype)
# 3. Establecer los Betas reales (Valores que queremos recuperar)
beta0 = 0.01 # Intercepto (Equivale a una tasa base de ~0.095)
beta2 = 1   # Efecto de X2 (Aumenta el riesgo)
beta1 = 0  # Efecto de X1 (Disminuye el riesgo)

# 4. Calcular Lambda individual usando la función de enlace logarítmica
# log(lambda) = beta0 + beta1*X1 + beta2*X2
log_lambda = beta0 + beta2*X2 + beta1*X1 
lambda_i = np.exp(log_lambda)

# 5. Simular tiempos de seguimiento (t_i) entre 0.5 y 5 años
tiempos = df_simulacion['Tiempo']

# 6. Simular número de eventos: Poisson(lambda_i * t_i)
eventos = np.random.poisson(lambda_i * tiempos)

# --- RECUPERACIÓN DE BETAS CON UN GLM (Modelo Lineal Generalizado) ---

# Creamos el DataFrame para el análisis
df_estudio = pd.DataFrame({
    'eventos': eventos,
    'X1': X1,
    'X2': X2,
    'log_t': np.log(tiempos) # Importante: log(tiempo) es el offset
})

# Definir variables independientes (añadiendo el intercepto)
X_modelo = sm.add_constant(df_estudio[['X1', 'X2']])

# Ajustar Modelo de Poisson
# El offset es vital porque: log(E[y]) = log(lambda * t) = log(lambda) + log(t)
modelo_glm = sm.GLM(df_estudio['eventos'], X_modelo, 
                    family=sm.families.Poisson(), 
                    offset=df_estudio['log_t']).fit()

# --- COMPARACIÓN DE RESULTADOS ---
resultados = pd.DataFrame({
    'Beta_Real': [beta0, beta1, beta2],
    'Beta_Recuperado': modelo_glm.params.values,
    'Error_Estandar': modelo_glm.bse.values
}, index=['Intercepto', 'X1 (Tratamiento)', 'X2 (Edad)'])

print("--- RESULTADOS DE LA RECUPERACIÓN ---")
print(resultados)
print("\n" + "="*30)
print("Resumen estadístico detallado:")
print(modelo_glm.summary())

Tipo de X1: object
Tipo de X2: object


TypeError: unsupported operand type(s) for +: 'float' and 'str'